<a href="https://colab.research.google.com/github/littlejacinthe/Style_Transfer/blob/main/Style_Transfer_STN_Updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

# Style Transfer with STN - Updated 2024

**Updates:**
- TensorFlow 2.x (latest stable version)
- Python 3.10+ compatible
- Updated librosa API
- Modern NumPy compatibility
- Fixed deprecated APIs and warnings

In [ ]:
# Install or upgrade required packages
!pip install --upgrade tensorflow numpy librosa matplotlib scikit-image scipy -q

In [ ]:
import tensorflow as tf
import librosa
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import os
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")
%matplotlib inline

## Audio Processing Functions

Updated functions for reading and processing audio files with modern librosa API

In [ ]:
# Configuration
N_FFT = 2048
HOP_LENGTH = 512
MAX_AUDIO_LENGTH = 10  # seconds

def read_audio_spectrum(filename, sr=None):
    """
    Read audio file and compute STFT magnitude spectrogram.
    
    Args:
        filename: Path to audio file
        sr: Sampling rate (default: librosa default 22050 Hz)
    
    Returns:
        S: Log-scaled magnitude spectrogram
        fs: Sampling frequency
    """
    try:
        # Load audio file
        x, fs = librosa.load(filename, sr=sr)
        
        # Limit audio length
        max_samples = int(fs * MAX_AUDIO_LENGTH)
        sig = x[:max_samples]
        
        # Compute STFT (modern librosa API uses n_fft and hop_length as keyword args)
        S = librosa.stft(sig, n_fft=N_FFT, hop_length=HOP_LENGTH)
        
        # Get magnitude and apply log scaling
        S_mag = np.abs(S)
        S_log = np.log1p(S_mag)  # log1p avoids log(0)
        
        # Limit frequency bins to first 430
        S_log = S_log[:, :430]
        
        return S_log, fs
    
    except Exception as e:
        print(f"Error reading audio file {filename}: {e}")
        return None, None

def plot_spectrogram(S, title="Spectrogram"):
    """
    Plot spectrogram.
    
    Args:
        S: Log-scaled magnitude spectrogram
        title: Title for the plot
    """
    plt.figure(figsize=(12, 4))
    plt.imshow(S, aspect='auto', origin='lower', cmap='viridis')
    plt.colorbar(label='Log Magnitude')
    plt.title(title)
    plt.xlabel('Time Frames')
    plt.ylabel('Frequency Bins')
    plt.tight_layout()
    plt.show()

## Audio File Setup

Define paths to content and style audio files

In [ ]:
# Audio file paths (for Colab)
CONTENT_FILENAME = '/content/classical_03.wav'
STYLE_FILENAME = '/content/jazz_05.wav'

# Check if files exist
content_exists = os.path.exists(CONTENT_FILENAME)
style_exists = os.path.exists(STYLE_FILENAME)

print(f"Content audio file exists: {content_exists}")
print(f"Style audio file exists: {style_exists}")

if content_exists and style_exists:
    print("\n--- Content Audio ---")
    display(Audio(CONTENT_FILENAME))
    
    print("\n--- Style Audio ---")
    display(Audio(STYLE_FILENAME))
else:
    print("\nWarning: Audio files not found. Make sure they are uploaded to /content/")

## Load and Visualize Spectrograms

Process audio files and display their spectrograms

In [ ]:
# Load spectrograms if files exist
if content_exists and style_exists:
    print("Loading spectrograms...")
    
    S_content, fs_content = read_audio_spectrum(CONTENT_FILENAME)
    S_style, fs_style = read_audio_spectrum(STYLE_FILENAME)
    
    if S_content is not None and S_style is not None:
        print(f"\nContent spectrogram shape: {S_content.shape}")
        print(f"Style spectrogram shape: {S_style.shape}")
        print(f"Sampling rate (content): {fs_content} Hz")
        print(f"Sampling rate (style): {fs_style} Hz")
        
        # Plot spectrograms
        plot_spectrogram(S_content, "Content Audio Spectrogram")
        plot_spectrogram(S_style, "Style Audio Spectrogram")
else:
    print("Skipping spectrogram loading - audio files not found")

## Spatial Transformer Network (STN) Model

Define the STN architecture for style transfer

In [ ]:
def create_stn_model(input_shape=(430, 430, 1)):
    """
    Create Spatial Transformer Network model.
    
    Args:
        input_shape: Input shape (height, width, channels)
    
    Returns:
        model: TensorFlow/Keras model
    """
    inputs = tf.keras.Input(shape=input_shape)
    
    # Localization network
    loc = tf.keras.layers.Conv2D(32, (5, 5), activation='relu', padding='same')(inputs)
    loc = tf.keras.layers.MaxPooling2D((2, 2))(loc)
    loc = tf.keras.layers.Conv2D(64, (5, 5), activation='relu', padding='same')(loc)
    loc = tf.keras.layers.MaxPooling2D((2, 2))(loc)
    loc = tf.keras.layers.Flatten()(loc)
    loc = tf.keras.layers.Dense(128, activation='relu')(loc)
    
    # Affine transformation parameters (6 for 2D affine)
    theta = tf.keras.layers.Dense(6, weights=[np.zeros((128, 6)), np.array([1., 0., 0., 0., 1., 0.], dtype=np.float32)])(loc)
    
    # Reshape theta for bilinear sampler
    theta = tf.keras.layers.Reshape((2, 3))(theta)
    
    # Note: Full STN implementation requires custom ops for spatial transformer
    # This is a simplified architecture
    
    model = tf.keras.Model(inputs=inputs, outputs=theta)
    return model

print("STN model definition ready")

## Data Preprocessing

Prepare spectrograms for model input

In [ ]:
def preprocess_spectrogram(S, target_shape=(430, 430)):
    """
    Preprocess spectrogram for model input.
    
    Args:
        S: Spectrogram (magnitude, log-scaled)
        target_shape: Target shape for resizing
    
    Returns:
        S_processed: Normalized and reshaped spectrogram
    """
    # Handle different input shapes
    if S.ndim == 2:
        # Pad or crop to target shape
        if S.shape[1] < target_shape[1]:
            # Pad if necessary
            S = np.pad(S, ((0, 0), (0, target_shape[1] - S.shape[1])), mode='constant')
        else:
            S = S[:, :target_shape[1]]
        
        # Normalize to [0, 1]
        S_min = S.min()
        S_max = S.max()
        if S_max > S_min:
            S = (S - S_min) / (S_max - S_min)
        else:
            S = np.zeros_like(S)
        
        # Add channel dimension
        S = np.expand_dims(S, axis=-1)
    
    return S.astype(np.float32)

# Test preprocessing
if content_exists and style_exists and S_content is not None:
    S_content_processed = preprocess_spectrogram(S_content)
    S_style_processed = preprocess_spectrogram(S_style)
    
    print(f"Processed content shape: {S_content_processed.shape}")
    print(f"Processed style shape: {S_style_processed.shape}")
    print(f"Processed content range: [{S_content_processed.min():.4f}, {S_content_processed.max():.4f}]")
    print(f"Processed style range: [{S_style_processed.min():.4f}, {S_style_processed.max():.4f}]")

## Model Training Setup

Configure and prepare for model training (placeholder for future implementation)

In [ ]:
# Training configuration
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001

# Create model
print("Creating STN model...")
model = create_stn_model(input_shape=(430, 430, 1))

# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='mse',
    metrics=['mae']
)

print("Model compiled successfully")
model.summary()

## Results and Visualization

Display results from style transfer

In [ ]:
print("Style Transfer Setup Complete!")
print("\nNext steps:")
print("1. Prepare your training data")
print("2. Define loss functions for style and content")
print("3. Train the model")
print("4. Apply style transfer to new audio")
print("\nNotebook successfully updated to TensorFlow 2.x!")